# Day 28：Document Loading & Text Splitting（RAG Step 1）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段（LangChain 与 RAG 实战）的第一天（Day 28）！今天我们将学习并实战 `Document`、`TextLoader`、`MarkdownLoader`、`CSVLoader`、`CharacterTextSplitter`、`RecursiveCharacterTextSplitter` 与 `MarkdownHeaderTextSplitter`，构建企业私域知识库接入的第一公里（`v0.3.0-day28`）。

## 1. 环境初始化与导入 `project_iterations/day_28` 模块
自动定位仓库根目录并将 `project_iterations/day_28` 加入 `sys.path`。

In [1]:
import sys
import tempfile
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd
for parent in [cwd, *cwd.parents]:
    if (parent / 'project_iterations' / 'day_28').exists():
        repo_root = parent
        break

day_dir = repo_root / 'project_iterations' / 'day_28'
if str(day_dir) not in sys.path:
    sys.path.insert(0, str(day_dir))

from zhishu_agent.document_loader import (
    CharacterTextSplitter,
    CSVLoader,
    Document,
    MarkdownHeaderTextSplitter,
    MarkdownLoader,
    RecursiveCharacterTextSplitter,
    TextLoader,
)

print('成功导入 Day 28 模块，路径:', day_dir)

成功导入 Day 28 模块，路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_28


## 2. 多格式文档加载器（TextLoader / MarkdownLoader / CSVLoader）
使用 `tempfile.TemporaryDirectory` 模拟企业内部的 `.txt` 制度文件、`.md` 技术手册与 `.csv` FAQ 表格。

In [2]:
with tempfile.TemporaryDirectory() as tmpdir:
    base = Path(tmpdir)
    txt_path = base / 'travel_policy.txt'
    txt_path.write_text(
        '星辰数科员工差旅报销规范。\n\n'
        '第一条：出差前需在OA系统提交审批。审批通过后方可预订机票与酒店。\n'
        '第二条：电子发票需在归来后7个工作日内上传。财务审核后3个工作日内打款。',
        encoding='utf-8',
    )

    md_path = base / 'architecture.md'
    md_path.write_text(
        '# 智枢平台架构手册\n\n'
        '## 1. 数据接入层\n支持 TXT、Markdown 与 CSV 文档加载。\n\n'
        '## 2. 文本切分层\n'
        '```python\n# 代码块里的井号注释不能被当成标题\nchunk_size = 500\n```\n'
        '推荐使用 RecursiveCharacterTextSplitter 保持语义完整。',
        encoding='utf-8',
    )

    csv_path = base / 'faq.csv'
    csv_path.write_text(
        'id,question,answer\n'
        'faq-1,如何申请VPN权限,请在OA门户提交IT工单\n'
        'faq-2,报销打款周期,审核通过后3个工作日内\n',
        encoding='utf-8',
    )

    txt_docs = TextLoader(txt_path).load()
    md_docs = MarkdownLoader(md_path).load()
    csv_docs = CSVLoader(csv_path, source_column='id').load()

    print('TXT Document:', txt_docs[0].metadata)
    print('MD  Document:', md_docs[0].metadata)
    print('CSV Documents 数量:', len(csv_docs), '| 第一条:', csv_docs[0].metadata)
    assert len(txt_docs) == 1
    assert md_docs[0].metadata['title'] == '智枢平台架构手册'
    assert len(csv_docs) == 2 and csv_docs[0].metadata['source'] == 'faq-1'

TXT Document: {'source': '/tmp/tmp1_udrmif/travel_policy.txt', 'file_name': 'travel_policy.txt', 'file_type': '.txt', 'loader': 'TextLoader'}
MD  Document: {'source': '/tmp/tmp1_udrmif/architecture.md', 'file_name': 'architecture.md', 'file_type': '.md', 'loader': 'MarkdownLoader', 'title': '智枢平台架构手册'}
CSV Documents 数量: 2 | 第一条: {'source': 'faq-1', 'file_name': 'faq.csv', 'file_type': '.csv', 'row': 0, 'loader': 'CSVLoader'}


## 3. 文本切分实战：CharacterTextSplitter vs RecursiveCharacterTextSplitter
验证多级分隔符 `["\n\n", "\n", "。", " ", ""]` 对中文段落与句子的保护效果，以及 `chunk_overlap` 重叠窗口。

In [3]:
sample_doc = Document(
    page_content=(
        '智枢平台第三阶段正式开启。\n\n'
        '文档加载器负责接入多源异构数据。文本切分器负责将长文档切分为语义完整的片段。最后交给向量库检索。'
    ),
    metadata={'source': 'intro.txt', 'dept': 'AI平台部'},
)

char_splitter = CharacterTextSplitter(separator='\n\n', chunk_size=40, chunk_overlap=8)
rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=35,
    chunk_overlap=8,
    separators=['\n\n', '\n', '。', ' ', ''],
)

char_chunks = char_splitter.split_documents([sample_doc])
rec_chunks = rec_splitter.split_documents([sample_doc])

print(f'CharacterTextSplitter 切分数量: {len(char_chunks)}')
for d in rec_chunks:
    print(f"[chunk#{d.metadata['chunk_index']}/{d.metadata['chunk_total']}] len={len(d.page_content)} -> {d.page_content}")

assert all(len(d.page_content) <= 35 for d in rec_chunks)
assert all(d.metadata['dept'] == 'AI平台部' for d in rec_chunks)

CharacterTextSplitter 切分数量: 3
[chunk#0/3] len=13 -> 智枢平台第三阶段正式开启。
[chunk#1/3] len=15 -> 文档加载器负责接入多源异构数据
[chunk#2/3] len=31 -> 文本切分器负责将长文档切分为语义完整的片段。最后交给向量库检索


## 4. 结构化切分：MarkdownHeaderTextSplitter 与两阶段切分
验证标题层级元数据注入、代码块 `#` 注释过滤，以及低级标题切换清理。

In [4]:
md_content = (
    '# 产品设计\n'
    '总体介绍。\n'
    '## 架构设计\n'
    '核心模块拆解。\n'
    '### 存储层\n'
    '使用 SQLite 与向量库。\n'
    '```python\n# 这是代码块注释，不能作为标题\nDB_PORT = 5432\n```\n'
    '## 部署指南\n'
    '使用 Docker Compose 启动服务。'
)

header_splitter = MarkdownHeaderTextSplitter()
header_docs = header_splitter.split_documents(
    [Document(page_content=md_content, metadata={'source': 'design.md'})]
)

for doc in header_docs:
    print('Metadata:', doc.metadata)
    print('Content :', repr(doc.page_content))
    print('-' * 50)

assert len(header_docs) == 4
assert 'Header 3' not in header_docs[-1].metadata
assert header_docs[-1].metadata['Header 2'] == '部署指南'
print('Day 28 Notebook 全部校验通过！')

Metadata: {'source': 'design.md', 'Header 1': '产品设计', 'chunk_index': 0, 'chunk_total': 4}
Content : '总体介绍。'
--------------------------------------------------
Metadata: {'source': 'design.md', 'Header 1': '产品设计', 'Header 2': '架构设计', 'chunk_index': 1, 'chunk_total': 4}
Content : '核心模块拆解。'
--------------------------------------------------
Metadata: {'source': 'design.md', 'Header 1': '产品设计', 'Header 2': '架构设计', 'Header 3': '存储层', 'chunk_index': 2, 'chunk_total': 4}
Content : '使用 SQLite 与向量库。\n```python\n# 这是代码块注释，不能作为标题\nDB_PORT = 5432\n```'
--------------------------------------------------
Metadata: {'source': 'design.md', 'Header 1': '产品设计', 'Header 2': '部署指南', 'chunk_index': 3, 'chunk_total': 4}
Content : '使用 Docker Compose 启动服务。'
--------------------------------------------------
Day 28 Notebook 全部校验通过！
